# Alpöge RH Replay — Autonomous Linux Volunteer Worker (Google Colab)

This notebook runs an **official, SHA-256–pinned Linux `x86_64` volunteer replay worker** for [`rohanarun/alpoge-rh-replay`](https://github.com/rohanarun/alpoge-rh-replay) using your **Google AI Ultra** Colab runtime (**0 AI tokens used**).

### Quick Start (30 seconds)
1. **Runtime (optional boost):** Click **Runtime → Change runtime type** → select **CPU** and toggle **High-RAM** on (if available).
2. **Add your GitHub token (`GH_TOKEN`):**
   - On your PC in PowerShell, run `gh auth token` and copy the token.
   - In Colab's left sidebar, click the **🔑 Secrets** icon → **Add new secret** → Name: `GH_TOKEN`, Value: *(paste token)* → toggle **Notebook access** ON. *(Or just run Cell 1 and paste it into the hidden prompt!)*
3. **Run All (`Runtime → Run all`):**
   - **Cell 1** verifies `x86_64` Linux, installs `elan` + `gh`, and holds `GH_TOKEN` **only in Python memory** (never exposed to Lean tactics or disk).
   - **Cell 2** runs `./verify.sh /content/rh-vol --prepare-only` (~3–6 min once per session on Google's gigabit network) to verify all 14 archive hashes, Lean `4.33.1` binary/library pins, and the 8,690 Mathlib cache files.
   - **Cell 3** continuously claims low-cost/warm-cache open modules, compiles & kernel-checks them (`leanchecker`), uploads verified receipts/bundles to `Sethumathavanpk/alpoge-rh-replay` Releases, and posts `Replay submission` comments to credit **`@Sethumathavanpk`** on the main README!


In [ ]:
import getpass
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess

assert platform.system() == 'Linux' and platform.machine() in ('x86_64', 'aarch64'), (
    f"Unsupported platform: {platform.system()} {platform.machine()} (need Linux x86_64/aarch64)"
)

mem_gib = 0.0
with open('/proc/meminfo') as f:
    for line in f:
        if line.startswith('MemTotal:'):
            mem_gib = int(line.split()[1]) / (1024 * 1024)
            break
disk_free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
print(f"Host: Linux {platform.machine()} | CPUs: {os.cpu_count()} | RAM: {mem_gib:.1f} GiB | Free disk: {disk_free_gib:.1f} GiB")
assert disk_free_gib >= 16.0, f"Need at least 16 GiB free disk on /content, found {disk_free_gib:.1f} GiB"

# Install gh CLI & elan if not already installed
if not shutil.which('gh'):
    print("Installing GitHub CLI (gh)...")
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'gh'], check=True)

elan_bin = Path.home() / '.elan/bin'
if not (elan_bin / 'elan').exists() and not shutil.which('elan'):
    print("Installing elan...")
    subprocess.run(
        'curl https://elan.lean-lang.org/elan-init.sh -sSf | sh -s -- -y --default-toolchain none',
        shell=True, check=True,
    )
if str(elan_bin) not in os.environ['PATH']:
    os.environ['PATH'] = f"{elan_bin}:{os.environ['PATH']}"

# Load GH_TOKEN into Python memory ONLY (never exported into global os.environ, so Lean tactics never see it)
os.environ.pop('GH_TOKEN', None)
os.environ.pop('GITHUB_TOKEN', None)
_GH_TOKEN = None
try:
    from google.colab import userdata
    _GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    _GH_TOKEN = None

if not _GH_TOKEN:
    _GH_TOKEN = getpass.getpass('Paste your GitHub token (from `gh auth token`): ').strip()
else:
    _GH_TOKEN = _GH_TOKEN.strip()

assert _GH_TOKEN, "A GitHub token is required."

def gh_api(*args, input_obj=None):
    env = {**os.environ, 'GH_TOKEN': _GH_TOKEN}
    res = subprocess.run(
        ['gh', *args],
        input=None if input_obj is None else json.dumps(input_obj),
        capture_output=True, text=True, env=env,
    )
    if res.returncode != 0:
        raise RuntimeError(f"gh {' '.join(args[:4])} failed: {res.stderr.strip()}")
    return res.stdout

me = json.loads(gh_api('api', 'user'))['login']
print(f"Authenticated with GitHub as @{me} (token isolated in controller memory only)!")


In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path('/content/alpoge-rh-replay')
VOL_DIR = Path('/content/rh-vol')
UPSTREAM_REPO = 'rohanarun/alpoge-rh-replay'
FORK_REPO = 'Sethumathavanpk/alpoge-rh-replay'

if not REPO_DIR.exists():
    print("Cloning rohanarun/alpoge-rh-replay...")
    subprocess.run(['git', 'clone', '--depth', '1', f'https://github.com/{UPSTREAM_REPO}.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)

lean_checkout = VOL_DIR / 'lean'
if not (lean_checkout / '.lake/packages/mathlib').is_dir():
    for attempt in range(1, 4):
        if VOL_DIR.exists():
            shutil.rmtree(VOL_DIR)
        print(f"=== Running ./verify.sh {VOL_DIR} --prepare-only (attempt {attempt}/3) ===")
        proc = subprocess.run(
            ['bash', str(REPO_DIR / 'verify.sh'), str(VOL_DIR), '--prepare-only'],
            cwd=str(REPO_DIR),
        )
        if proc.returncode == 0:
            # Free compressed archive tarballs once extracted & verified
            shutil.rmtree(VOL_DIR / 'parts', ignore_errors=True)
            print("Volume prepared and SHA-256 verified!")
            break
        print(f"Prepare attempt {attempt} exited with {proc.returncode}; retrying...")
    else:
        raise RuntimeError("Failed to prepare replay volume after 3 attempts.")
else:
    print(f"Reusing warm SHA-verified volume at {lean_checkout}!")


In [ ]:
from collections import Counter
import csv
import gzip
import json
import os
from pathlib import Path
import random
import re
import subprocess
import sys
import time

# === CONFIGURE HOW MANY MODULES TO VERIFY & SUBMIT IN THIS RUN ===
MAX_MODULES_TO_COMPLETE = 15   # Increase to 50+ for overnight Colab Background Execution!
MAX_UNBUILT_CLOSURE = 18       # Max unbuilt project modules per claim (warm cache makes deeper modules qualify!)
LEAN_THREADS = str(max(2, min(8, (os.cpu_count() or 2))))

sys.path.insert(0, str((REPO_DIR / 'tracker').resolve()))
import replay_evidence as evidence
import tasks
import worker_build

def refresh_ledger():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'checkout', 'origin/main', '--',
         'tracker/completed.tsv', 'tracker/modules.jsonl.gz', 'tracker/summary.json'],
        check=True,
    )

def load_rows():
    done = {}
    comp = REPO_DIR / 'tracker/completed.tsv'
    if comp.exists():
        with comp.open(encoding='utf-8') as f:
            done = {r['id']: r['evidence'] for r in csv.DictReader(f, delimiter='\t')}
    rows = {}
    with gzip.open(REPO_DIR / 'tracker/modules.jsonl.gz', 'rt', encoding='utf-8') as s:
        for line in s:
            r = json.loads(line)
            if r['id'] in done:
                r['state'] = done[r['id']]
            rows[r['id']] = r
    return rows

def fetch_open_claims():
    """Use GraphQL cursor pagination so we never stop early on 99-item pages or hit the 3,000-issue REST limit."""
    query = '''query($cursor:String) {
      repository(owner:"rohanarun", name:"alpoge-rh-replay") {
        issues(states:OPEN, first:100, after:$cursor) {
          pageInfo { hasNextPage endCursor }
          nodes { title }
        }
      }
    }'''
    busy = set()
    cursor = None
    pages = 0
    while True:
        args = ['api', 'graphql', '-f', f'query={query}']
        if cursor:
            args += ['-f', f'cursor={cursor}']
        data = json.loads(gh_api(*args))['data']['repository']['issues']
        pages += 1
        for node in data['nodes']:
            if not node:
                continue
            m = re.fullmatch(r'Claim (M\d{6}|P\d{4})', node.get('title') or '')
            if m:
                busy.add(m.group(1))
        if not data['pageInfo']['hasNextPage']:
            break
        cursor = data['pageInfo']['endCursor']
    print(f"Loaded {len(busy)} open claims across {pages} GraphQL pages.", flush=True)
    return busy

def is_still_unclaimed(mid, pkt):
    q = f'repo:{UPSTREAM_REPO} is:issue is:open "{mid}" OR "{pkt}" in:title'
    items = json.loads(gh_api('api', '-X', 'GET', 'search/issues', '-f', f'q={q}')).get('items', [])
    for it in items:
        t = it.get('title', '')
        if t in (f'Claim {mid}', f'Claim {pkt}'):
            return False
    return True

def warm_cached_ids(rows):
    build_lib = (VOL_DIR / 'lean/.lake/build/lib/lean')
    if not build_lib.is_dir():
        return set()
    by_mod = {r['module']: r['id'] for r in rows.values()}
    cached = set()
    for p in build_lib.rglob('*.olean'):
        if p.is_file():
            mod = '.'.join(p.relative_to(build_lib).with_suffix('').parts)
            if mod in by_mod:
                cached.add(by_mod[mod])
    return cached

def pick_candidates(rows, busy, cached_ids, pool_size=160):
    consumers = Counter()
    for r in rows.values():
        if r['state'] == 'open':
            for dep in r['imports']:
                consumers[dep] += 1

    cands = []
    for mid, r in rows.items():
        if r['state'] != 'open' or mid in busy or r['packet'] in busy:
            continue
        if r['depth'] > MAX_UNBUILT_CLOSURE + 5 and not cached_ids:
            continue
        unbuilt = set()
        seen = {mid}
        stack = [mid]
        too_big = False
        while stack:
            cur = stack.pop()
            if cur not in cached_ids:
                unbuilt.add(cur)
                if len(unbuilt) > MAX_UNBUILT_CLOSURE:
                    too_big = True
                    break
            for dep in rows[cur]['imports']:
                if dep not in seen:
                    seen.add(dep)
                    stack.append(dep)
        if not too_big:
            cands.append((len(unbuilt), r['depth'], -consumers[mid], mid, r['packet'], r['module']))
    cands.sort()
    pool = cands[:pool_size]
    random.shuffle(pool)  # Avoid thundering-herd races with deterministic volunteer bots!
    return pool[:25]

def claim_and_wait_grant(target, pkt, owner, timeout_s=3600):
    if not is_still_unclaimed(target, pkt):
        print(f"  {target} ({pkt}) was just claimed moments ago; skipping before opening issue!", flush=True)
        return None
    body = (
        f'I am working on {target}. '
        f'I will attach the input hashes, build transcript and receipt, or close this claim if I stop.'
    )
    created = json.loads(gh_api(
        'api', '--method', 'POST', f'repos/{UPSTREAM_REPO}/issues',
        '--input', '-', input_obj={'title': f'Claim {target}', 'body': body},
    ))
    num = created['number']
    print(f"Opened Claim {target}: https://github.com/{UPSTREAM_REPO}/issues/{num} — waiting for bot grant...", flush=True)
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        time.sleep(12)
        iss = json.loads(gh_api('api', f'repos/{UPSTREAM_REPO}/issues/{num}'))
        labels = {l['name'] for l in iss.get('labels', [])}
        comments = json.loads(gh_api('api', f'repos/{UPSTREAM_REPO}/issues/{num}/comments?per_page=100'))
        if 'claim-conflict' in labels or any(
            c['user']['login'] == 'github-actions[bot]' and c['body'].startswith('No grant.')
            for c in comments
        ):
            print(f"  Conflict on #{num}; closing cleanly and picking next...", flush=True)
            gh_api('api', '--method', 'PATCH', f'repos/{UPSTREAM_REPO}/issues/{num}',
                   '--input', '-', input_obj={'state': 'closed'})
            return None
        if 'claim-granted' in labels and any(
            c['user']['login'] == 'github-actions[bot]' and c['body'].startswith(f'Granted `{target}`.')
            for c in comments
        ):
            print(f"  Granted `{target}` on #{num}!", flush=True)
            return num
    print(f"  Grant wait timed out on #{num}; closing cleanly...", flush=True)
    gh_api('api', '--method', 'PATCH', f'repos/{UPSTREAM_REPO}/issues/{num}',
           '--input', '-', input_obj={'state': 'closed'})
    return None

def publish_release(fork, target, issue_num, files):
    tag = f'replay-{target}-issue-{issue_num}'
    route = f'repos/{fork}/releases/tags/{tag}'
    env = {**os.environ, 'GH_TOKEN': _GH_TOKEN}
    probe = subprocess.run(['gh', 'api', route], capture_output=True, text=True, env=env)
    if probe.returncode != 0:
        if '(HTTP 404)' not in probe.stderr and 'Not Found' not in probe.stderr:
            raise RuntimeError(probe.stderr.strip())
        subprocess.run([
            'gh', 'release', 'create', tag, '--repo', fork,
            '--title', target, '--notes', f'Replay evidence for {UPSTREAM_REPO}#{issue_num}.'
        ], check=True, env=env)
        remote = json.loads(gh_api('api', route))
    else:
        remote = json.loads(probe.stdout)

    assets = {a['name']: a for a in remote.get('assets', [])}
    missing = [str(p) for p in files if p.name not in assets]
    if missing:
        subprocess.run(['gh', 'release', 'upload', tag, '--repo', fork, *missing], check=True, env=env)
        remote = json.loads(gh_api('api', route))
        assets = {a['name']: a for a in remote.get('assets', [])}

    links = []
    for p in files:
        asset = assets[p.name]
        exp = 'sha256:' + tasks.digest(p)
        if asset.get('state') != 'uploaded':
            raise RuntimeError(f'Asset not uploaded cleanly: {p.name}')
        if asset.get('digest') and asset['digest'] != exp:
            raise RuntimeError(f'Digest mismatch on {p.name}')
        links.append((p.name, asset['browser_download_url'], tasks.digest(p)))
    return tag, links

owner = json.loads(gh_api('api', 'user'))['login']
receipts_root = VOL_DIR / 'receipts'
completed_submissions = []
os.environ['LEAN_NUM_THREADS'] = LEAN_THREADS

while len(completed_submissions) < MAX_MODULES_TO_COMPLETE:
    refresh_ledger()
    rows = load_rows()
    busy = fetch_open_claims()
    cached = warm_cached_ids(rows)
    cands = pick_candidates(rows, busy, cached, pool_size=160)
    if not cands:
        raise RuntimeError("No open candidates within MAX_UNBUILT_CLOSURE; increase MAX_UNBUILT_CLOSURE!")

    target = None
    issue_num = None
    for unbuilt_cnt, depth, neg_cons, mid, pkt, mod in cands:
        print(f"Trying {mid} ({pkt}, depth={depth}, unbuilt_closure={unbuilt_cnt}, unblocks={-neg_cons}): {mod}", flush=True)
        issue_num = claim_and_wait_grant(mid, pkt, owner)
        if issue_num is not None:
            target = mid
            break
    if target is None:
        continue

    out_dir = receipts_root / f'{target}-{issue_num}'
    print(f"Building & verifying {target} (LEAN_NUM_THREADS={LEAN_THREADS})...", flush=True)
    code = worker_build.build(target, VOL_DIR / 'lean', out_dir)
    if code != 0:
        raise RuntimeError(f"Build exited with {code} on {target}; inspect {out_dir / 'build.log'} before closing #{issue_num}")

    by_mod = {r['module']: r for r in rows.values()}
    targets = tasks.select(by_mod.values(), target)
    verified = evidence.verify(out_dir, targets, by_mod, VOL_DIR / 'lean')
    receipt = json.loads((out_dir / 'receipt.json').read_text(encoding='utf-8'))

    files = [out_dir / 'receipt.json', out_dir / 'build.log', out_dir / 'targets.tar.gz']
    if (out_dir / evidence.REPLAY_FILE).is_file():
        files.append(out_dir / evidence.REPLAY_FILE)

    tag, asset_links = publish_release(FORK_REPO, target, issue_num, files)
    links_md = ' · '.join(f'[{name}]({url})' for name, url, _ in asset_links)
    sha_lines = [f'{sha}  {name}' for name, _, sha in asset_links]
    for tid, osha in sorted(receipt['objects'].items()):
        sha_lines.append(f'{osha}  {tid} (.olean)')

    rebuilt_cnt = len(verified['rebuilt_project_modules'])
    comment_body = (
        f"Replay submission\n\n"
        f"{target}, Lake exit 0 on {verified['identity']['platform']}. "
        f"Of {len(verified['tasks'])} requested targets, Lake built {len(verified['built_targets'])} "
        f"and the kernel replayed {len(verified['kernel_replayed_targets'])} from cache "
        f"({rebuilt_cnt} prerequisite project imports built locally). "
        f"Elapsed Lake time {receipt['elapsed_seconds']} s.\n\n"
        f"{links_md}\n\n"
        f"Bundle SHA-256 `{receipt['bundle_sha56' if 'bundle_sha56' in receipt else 'bundle_sha256']}`.\n\n"
        f"```text\n" + '\n'.join(sha_lines) + "\n```\n"
    )

    iss_now = json.loads(gh_api('api', f'repos/{UPSTREAM_REPO}/issues/{issue_num}'))
    lbls_now = {l['name'] for l in iss_now.get('labels', [])}
    if iss_now['state'] != 'open' or 'claim-granted' not in lbls_now:
        raise RuntimeError(f"Grant changed before submit on #{issue_num}: state={iss_now['state']} labels={lbls_now}")

    posted = json.loads(gh_api(
        'api', '--method', 'POST', f'repos/{UPSTREAM_REPO}/issues/{issue_num}/comments',
        '--input', '-', input_obj={'body': comment_body},
    ))
    sub_url = posted['html_url']
    completed_submissions.append((target, issue_num, sub_url))
    print(f"[{len(completed_submissions)}/{MAX_MODULES_TO_COMPLETE}] SUBMITTED {target}: {sub_url}", flush=True)

print("\n=== ALL REQUESTED REPLAY SUBMISSIONS COMPLETED! ===")
for t, n, u in completed_submissions:
    print(f"  - {t} (#{n}): {u}")
